# Person 5 — Random Forest Classifier & Model Evaluation Pipeline

Pipeline Responsibility: Model Training & Held-Out Evaluation\nModel Assignment: Random Forest Classifier

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.ensemble import RandomForestClassifier

print("--- Person 5: Random Forest Training & Held-Out Evaluation (Winning Model) ---")

rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=SEED,
    n_jobs=2
)

start_time = time.perf_counter()
rf_model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = rf_model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

# Bootstrap 95% Confidence Interval for Macro F1
np.random.seed(SEED)
boot_f1s = []
n_samples = len(y_te)
for _ in range(1000):
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    boot_f1s.append(f1_score(y_te[idx], preds[idx], average='macro', zero_division=0))

ci_lower, ci_upper = np.percentile(boot_f1s, [2.5, 97.5])

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")
print(f"95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print("Confusion Matrix:\n", cm)

importances = rf_model.feature_importances_
top_20_feats = np.argsort(importances)[-20:][::-1].tolist()

metrics = {
    "model_name": "Random Forest (Pipeline Winner)",
    "pipeline_stage": "Final Fit & Held-Out Evaluation",
    "n_estimators": 200,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "bootstrap_95_ci": [float(ci_lower), float(ci_upper)],
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "top_20_features": top_20_feats,
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "random_forest_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(rf_model, OUTPUT_DIR / "random_forest_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)


--- Person 5: Random Forest Training & Held-Out Evaluation (Winning Model) ---


Accuracy: 0.9556
Macro F1: 0.9554
95% Bootstrap CI: [0.9216, 0.9832]
Confusion Matrix:
 [[92  0]
 [ 8 80]]
Saved outputs to: D:\SLIIT\projectr\Dataset_Train\parts\random_forest\outputs
